# AI-MeshOptimizer -- Training Notebook

Trains the MeshGNN edge-importance model used by **Feature-Weighted QEM Simplification**
(an open-source, learned alternative to ZRemesher).

Pipeline this notebook runs:

```
High Poly Mesh -> Feature Extraction -> GNN -> Edge Importance -> Feature-Weighted QEM -> Low Poly Mesh
```

Designed to run end-to-end on **Google Colab Free (T4 GPU)**, no paid services required.

1. Check GPU
2. Install dependencies
3. Clone the repository
4. Prepare a dataset (quick synthetic set by default, or your own ABC / Objaverse / Thingi10K meshes)
5. Run preprocessing (`generate_pairs.py`)
6. Train the model (`train.py`) -- saves `best_model.pt`
7. Plot training curves
8. Run inference on a sample mesh and print a quality report


## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Colab -> Runtime > Change runtime type > T4 GPU.")


## 2. Install dependencies

Colab already ships a CUDA build of PyTorch, so we only add the packages this project needs
on top of it. `torch-geometric` (>=2.4) works with plain `pip install` and no longer requires
the old `torch-scatter`/`torch-sparse` wheel-matching dance for the layers used here
(`GraphConv`, `GATConv`).

In [ ]:
!pip install -q torch-geometric
!pip install -q trimesh open3d pyfqmr networkx rtree tqdm matplotlib plyfile

import torch, torch_geometric
print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)


## 3. Get the project into Colab

Two ways to do this -- pick whichever applies:

- **Method A -- GitHub.** If you've pushed `AI-MeshOptimizer/` to your own
  GitHub repo, set `REPO_URL` below (a real URL, no `<...>` placeholders --
  those get misread as shell redirection and fail with a confusing
  `bash: ...: No such file or directory` error).
- **Method B -- zip upload (default, no GitHub needed).** Leave `REPO_URL`
  empty. Zip your local `AI-MeshOptimizer/` folder, run the cell, and pick the
  `.zip` file when the upload widget appears.

In [ ]:
import os

PROJECT_DIR = "/content/AI-MeshOptimizer"
REPO_URL = ""  # e.g. "https://github.com/yourname/AI-MeshOptimizer.git" -- leave empty to upload a zip instead

def _looks_like_project(path):
    return os.path.exists(os.path.join(path, "inference", "remesh.py"))

if _looks_like_project(PROJECT_DIR):
    print("Project already present at", PROJECT_DIR)
elif REPO_URL.strip():
    !git clone "{REPO_URL}" "{PROJECT_DIR}"
else:
    import zipfile
    import shutil
    from google.colab import files

    print("REPO_URL is empty -- upload a .zip of your local AI-MeshOptimizer/ folder:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

    extract_dir = "/content/_ai_meshoptimizer_extracted"
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_dir)

    # handle both "zip contains AI-MeshOptimizer/ as a subfolder" and
    # "zip contains the project files directly at the top level"
    entries = os.listdir(extract_dir)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        shutil.move(os.path.join(extract_dir, entries[0]), PROJECT_DIR)
    else:
        shutil.move(extract_dir, PROJECT_DIR)

assert _looks_like_project(PROJECT_DIR), (
    f"'{PROJECT_DIR}' doesn't look like the AI-MeshOptimizer project "
    "(inference/remesh.py not found). Fix REPO_URL or re-upload the correct zip."
)

%cd {PROJECT_DIR}
import sys
sys.path.append(PROJECT_DIR)
print("Ready at", os.getcwd())


## 4. Prepare the dataset

**Option A (default, no download needed):** generate a small synthetic set of primitive
meshes (spheres, boxes, cylinders, tori, capsules at several subdivision levels) so the
whole pipeline can be smoke-tested immediately on Colab Free.

**Option B (real training data):** point `RAW_DIR` at your own high-poly meshes instead --
e.g. a subset of the [ABC Dataset](https://deep-geometry.github.io/abc-dataset/),
[Objaverse](https://objaverse.allenai.org/), or [Thingi10K](https://ten-thousand-models.appspot.com/),
downloaded to Google Drive and mounted below. Free-tier Colab disk/session limits mean you
should start with a few hundred meshes, not the full multi-million-model corpora.

In [ ]:
# Option B: uncomment to mount Drive and use your own dataset directory instead
# from google.colab import drive
# drive.mount('/content/drive')
# RAW_DIR = "/content/drive/MyDrive/AI-MeshOptimizer/dataset/raw"

RAW_DIR = "dataset/raw"
os.makedirs(RAW_DIR, exist_ok=True)

if len(os.listdir(RAW_DIR)) == 0:
    print("No meshes found in", RAW_DIR, "-- generating a small synthetic starter set...")
    import trimesh
    import numpy as np

    def save(mesh, name):
        mesh.export(os.path.join(RAW_DIR, name))

    for i, sub in enumerate([2, 3, 4]):
        save(trimesh.creation.icosphere(subdivisions=sub), f"sphere_{i}.obj")
    for i, sec in enumerate([16, 32, 48]):
        save(trimesh.creation.cylinder(radius=1.0, height=2.0, sections=sec), f"cylinder_{i}.obj")
    for i, sub in enumerate([1, 2, 3]):
        save(trimesh.creation.box(extents=[2, 1, 1]).subdivide().subdivide(), f"box_{i}.obj")
    save(trimesh.creation.torus(major_radius=1.0, minor_radius=0.35), "torus.obj")
    save(trimesh.creation.capsule(radius=0.5, height=1.5), "capsule.obj")

print(f"{len(os.listdir(RAW_DIR))} raw meshes in {RAW_DIR}")


## 5. Run preprocessing

Extracts per-vertex/per-edge features, computes QEM-based ground-truth edge-importance
labels, generates a low-poly target with `pyfqmr` for Chamfer/normal supervision, and saves
one `torch_geometric.data.Data` graph per mesh under `dataset/processed/`.

In [ ]:
!python preprocessing/generate_pairs.py \
    --input_dir dataset/raw \
    --processed_dir dataset/processed \
    --pairs_dir dataset/pairs \
    --reduction_ratio 0.1 \
    --min_faces 50


## 6. Inspect the training dataset

In [ ]:
from training.dataset import MeshPairDataset

dataset = MeshPairDataset("dataset/processed")
print(f"{len(dataset)} graphs")
g = dataset[0]
print(g)
print("node features:", g.x.shape, "edge_index:", g.edge_index.shape, "edge_attr:", g.edge_attr.shape)
print("target_vertices:", g.target_vertices.shape)


## 7. Train the model

Parameters match the project spec: `epochs=100`, `batch_size=8`, `lr=0.0001`, `AdamW` +
`CosineAnnealingLR`. On Colab Free with a small dataset this finishes in minutes; scale
`epochs`/dataset size to your session budget.

In [ ]:
!python training/train.py \
    --processed_dir dataset/processed \
    --checkpoint_dir checkpoints \
    --epochs 100 \
    --batch_size 8 \
    --lr 0.0001 \
    --val_split 0.15 \
    --device auto


## 8. Plot training curves

`train.py` already saved `checkpoints/training_curves.png` (loss, accuracy, edge-prediction precision/recall/F1) -- display it here.

In [ ]:
from IPython.display import Image, display
display(Image(filename="checkpoints/training_curves.png"))


## 9. Run inference on a sample mesh

Uses the saved `checkpoints/best_model.pt` to predict edge importance, then runs
Feature-Weighted QEM simplification down to `--target_faces`.

In [ ]:
import glob

sample_mesh = sorted(glob.glob("dataset/raw/*"))[0]
print("Using sample mesh:", sample_mesh)

!python inference/remesh.py {sample_mesh} /content/output_low.obj \
    --target_faces 200 \
    --checkpoint checkpoints/best_model.pt \
    --compare


## 10. Visualize before / after (matplotlib)

In [ ]:
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def plot_mesh(ax, mesh, title):
    tris = mesh.vertices[mesh.faces]
    coll = Poly3DCollection(tris, alpha=0.85, edgecolor="k", linewidths=0.15)
    coll.set_facecolor((0.4, 0.6, 0.9))
    ax.add_collection3d(coll)
    bounds = mesh.vertices
    ax.set_xlim(bounds[:, 0].min(), bounds[:, 0].max())
    ax.set_ylim(bounds[:, 1].min(), bounds[:, 1].max())
    ax.set_zlim(bounds[:, 2].min(), bounds[:, 2].max())
    ax.set_title(f"{title}\n{len(mesh.faces)} faces")

original = trimesh.load(sample_mesh, process=False, force="mesh")
simplified = trimesh.load("/content/output_low.obj", process=False, force="mesh")

fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_mesh(ax1, original, "Original (High Poly)")
plot_mesh(ax2, simplified, "AI-MeshOptimizer (Low Poly)")
plt.tight_layout()
plt.show()


## Notes

- `checkpoints/best_model.pt` is the artifact you need for `inference/remesh.py`.
- To train on real data, drop meshes into `dataset/raw/` (or point `RAW_DIR` at a Drive
  folder) before step 5 -- everything downstream is identical.
- pyfqmr is only used to generate the *training* low-poly targets; at inference time,
  edge collapse is driven by `utils/qem.py`'s own weighted simplifier so the GNN's
  per-edge importance can actually bias which edges survive.